# Lab 05 · Gemini learned embeddings (optional API)

Module F • **Optional Google API** (may incur charges/quotas). Do the conceptual quiz without an API key. Gemini Embedding 001 is text-only; Embedding 2 is multimodal.

**Workflow:** read → predict → run → edit the marked variable → rerun → answer the linked questions in `assessments/quiz.html`.  \n**No answer needs a screenshot.** Keep your notebook output visible while taking the quiz.

# Your task board · Run first, answer later

Use the teaching cells below to determine the answers. For an **EDIT** task, change the marked variable and rerun the specified cell. The hints and full answers are intentionally **open at the end of this notebook**. Try the questions first.

**F1 · MCQ · after cell 05 · F1** — Which Gemini model from the original uploaded notebooks is text-only?
  - **A.** gemini-embedding-001
  - **B.** gemini-embedding-2
  - **C.** both text-only
  - **D.** neither

**F2 · NAT · after cell 05 · F1** — Which requested dimensionality does the Gemini 001 optional cell set?  
NAT: `_________`

**F3 · MCQ · after cell 05 · F1** — For retrieval with Gemini 001, which task types should you pair?
  - **A.** CLUSTERING for both
  - **B.** RETRIEVAL_DOCUMENT and RETRIEVAL_QUERY
  - **C.** SEMANTIC_SIMILARITY and CODE_EXECUTION
  - **D.** No model/task choice matters

**F4 · MCQ · after cell 05 · F3** — Can you directly compare Gemini 001 vectors with Gemini Embedding 2 vectors?
  - **A.** Yes if both have 768 dimensions
  - **B.** Yes after PCA
  - **C.** No, their learned spaces are incompatible
  - **D.** Only for cats

**F5 · MCQ · after cell 05 · F3** — Which statement about Gemini Embedding 2 is correct?
  - **A.** It is text-only
  - **B.** It always preserves audio in video files
  - **C.** It can embed text and images in a shared learned space
  - **D.** It is the same as PCA

**F6 · MCQ · after cell 05 · F0** — Which practice protects the API key?
  - **A.** Paste it into a committed notebook
  - **B.** Save in Colab Secrets or a local environment variable
  - **C.** Include it in submitted screenshots
  - **D.** Log it to show the TA

---

**Assessment:** [Interactive self-check](../assessments/quiz.html?module=F) · [Full question sheet](../assessments/QUESTION_SHEET.md). You can learn without grading; 70% unlocks the next quiz on the portal.

**Official documentation:** [Gemini embeddings](https://ai.google.dev/gemini-api/docs/embeddings) · [Gemini Embedding 2](https://ai.google.dev/gemini-api/docs/models/gemini-embedding-2).

In [ ]:
from pathlib import Path
import numpy as np
from IPython.display import display

def find_workshop_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/embedding_workshop')]
    for p in candidates:
        if (p / 'data' / 'field_notes.md').is_file():
            return p
    raise FileNotFoundError('Cannot find data/field_notes.md. Unzip the complete workshop, not just one notebook. Check README.md.')
ROOT = find_workshop_root()
print('Workshop root:', ROOT)


## F0 · Safe API key setup
Use Colab → Secrets → `GEMINI_API_KEY` or a local environment variable. **Never** paste a key into the notebook, screenshots or GitHub. If missing, the rest of the lab prints `SKIPPED` instead of failing.

In [ ]:
%pip -q install 'google-genai>=1.0' 

In [ ]:
import os
api_key=os.getenv('GEMINI_API_KEY')
if not api_key:
    try:
        from google.colab import userdata
        api_key=userdata.get('GEMINI_API_KEY')
    except Exception:
        api_key=None
HAS_KEY=bool(api_key)
print('Gemini access:', 'enabled' if HAS_KEY else 'SKIPPED — key not configured')

## F1 · Real text embeddings using gemini-embedding-001
Use matching task types for search: documents → `RETRIEVAL_DOCUMENT`; query → `RETRIEVAL_QUERY`. `SEMANTIC_SIMILARITY` is for symmetric sentence comparison rather than optimizing query/document retrieval. 768D non-default Gemini 001 outputs should be normalized manually for cosine-based workflows.

In [ ]:
from google import genai
from google.genai import types
texts=['The cat sits on a table.','A kitten rests on furniture.','A smartphone takes photos.']
if HAS_KEY:
    client=genai.Client(api_key=api_key)
    docs_result=client.models.embed_content(model='gemini-embedding-001',contents=texts,
              config=types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT',output_dimensionality=768))
    doc_vectors=np.array([e.values for e in docs_result.embeddings],dtype=float)
    doc_vectors=doc_vectors/np.linalg.norm(doc_vectors,axis=1,keepdims=True)
    print('doc_shape =',doc_vectors.shape)
    print('first_vector_norm =',round(float(np.linalg.norm(doc_vectors[0])),5))
else:
    print('SKIPPED: no API call or bill incurred.')

## F2 · Query the compatible text space

In [ ]:
if HAS_KEY:
    query_result=client.models.embed_content(model='gemini-embedding-001',contents='Where is the feline?',
                config=types.EmbedContentConfig(task_type='RETRIEVAL_QUERY',output_dimensionality=768))
    q=np.array(query_result.embeddings[0].values,dtype=float);q=q/np.linalg.norm(q)
    scores=doc_vectors@q
    print(sorted(zip(texts,np.round(scores,4)),key=lambda pair:-pair[1]))
else:
    print('SKIPPED: no API key.')

## F3 · Gemini Embedding 2: text + image in ONE learned space
The reference notebooks used **001**, which is text-only. The official Gemini Embedding 2 API supports text, image, audio, video and PDFs. For an image, the SDK sends bytes with a MIME type. This optional experiment makes **two separate embeddings**, not one aggregated embedding. Verify current model availability in Google AI docs; account/API access may vary.

In [ ]:
if HAS_KEY:
    image_bytes=(ROOT/'data/cat_illustration.png').read_bytes()
    model='gemini-embedding-2' # If your region only exposes a preview, check official model docs.
    try:
        text_out=client.models.embed_content(model=model,contents='An orange cat on a table',
                  config=types.EmbedContentConfig(output_dimensionality=768))
        image_out=client.models.embed_content(model=model,
                  contents=types.Content(parts=[types.Part.from_bytes(data=image_bytes,mime_type='image/png')]),
                  config=types.EmbedContentConfig(output_dimensionality=768))
        vt=np.asarray(text_out.embeddings[0].values,dtype=float)
        vi=np.asarray(image_out.embeddings[0].values,dtype=float)
        score=float(vt@vi/(np.linalg.norm(vt)*np.linalg.norm(vi)))
        print('text_dims =',len(vt),'image_dims =',len(vi),'crossmodal_cosine =',round(score,4))
    except Exception as exc:
        print('Provider/API error — check current model ID and quota:',type(exc).__name__,str(exc)[:240])
else:
    print('SKIPPED: use slides and the conceptual F quiz.')

## F4 · Independent judgment
1. What changes if you swap to a different embedding model? (Re-embed the corpus; spaces cannot be mixed.)
2. Does PCA make the vectors *three-dimensional embeddings*? (No: it is a 3D projection for display.)
3. Does the video representation automatically include its audio track? (Not necessarily; see the model’s video-specific processing rules.)

Finish **F1–F6** in the quiz.

## Hints · Use these only after attempting the tasks

- **F1:** Compare model input types in the two official model pages.
- **F2:** Find `output_dimensionality` in the optional request.
- **F3:** Use retrieval-specific document/query tasks for the 001 model.
- **F4:** Check whether the model families are trained in the same space.
- **F5:** Check the supported modalities for Gemini Embedding 2.
- **F6:** Look at the key-loading cell; it must not hard-code credentials.

These are learning hints, not the solutions. Scroll down for the fully open answer key.


## Open answers · Full worked key and explanations

The answers below refer to the **supplied dataset and stated default/EDIT settings**. If you changed additional inputs, your results may differ. Compare the experiment and formula—not just the number.

**F1: A — gemini-embedding-001**  
Gemini Embedding 001 supports text input, while Embedding 2 extends to multimodal.

**F2: 768**  
The EmbedContentConfig sets output_dimensionality=768.

**F3: B — RETRIEVAL_DOCUMENT and RETRIEVAL_QUERY**  
The asymmetric document/query task pair is designed for retrieval.

**F4: C — No, their learned spaces are incompatible**  
Same dimensionality does not make two learned spaces compatible; re-embed on migration.

**F5: C — It can embed text and images in a shared learned space**  
Gemini Embedding 2 is a multimodal embedding model.

**F6: B — Save in Colab Secrets or a local environment variable**  
Keys should be supplied securely and not committed or printed.

### Extension challenge — Interview explanation

**Challenge:** A team wants to mix Embedding 001 and Embedding 2 vectors after migration. What should you tell them and why?

**Self-check:** Explain the trade-off aloud in 45 seconds using a concrete example.
